# Benchmarking & Evaluations · Edge-Computing Notebook

In the Edge AI lab you logged raw inference latency. Logging numbers is not the same as **understanding** them. Benchmarking turns a pile of measurements into answers: How fast is it, really? How consistent? What does it cost in resources? Which configuration is best for this device?

```text
Latency logs  ->  Statistics (p50 / p95 / p99, FPS)
              ->  Resource cost (CPU, GPU, memory, power)
              ->  Configuration sweep (which setup wins)
              ->  Benchmark report
```

**You will:**
- compute the full latency distribution (min / p50 / mean / p95 / p99 / max) from your logs
- capture resource cost with `tegrastats` while inference runs
- sweep model size and compute device, one variable at a time
- summarize the sweep into a CSV results table
- test sustained load for thermal drift, and write it all up in a benchmark report

The main idea: an edge system is judged by evidence, not by guesses. A benchmark is a **repeatable** measurement under **controlled** conditions, reported with the **full distribution**, not just a best-case average.

## How this notebook works · cells and a Jupyter terminal

Analysis steps run once and return — those are ordinary notebook cells. Two steps run for many minutes or stream forever: the resource-cost capture (inference in one shell, `tegrastats` sampling in another) and the 400-run sustained test. For those, this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal — **File ▸ New ▸ Terminal** (or the Terminal tile in the Launcher) — and run the given command. Part 4 needs **two terminals** side by side.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell below writes `~/benchmarkLab/labEnv.sh`, and every helper script starts with `source ~/benchmarkLab/labEnv.sh` — the notebook and your terminals agree on `USER` and `DEVICE_ID` with no manual step.

**Requirements:** the latency logs and `latencyLogger.py` from the Edge AI lab (Lab 5). Parts 4, 5, and 7 run inference, so the kernel/terminal needs `ultralytics` — the same environment as Lab 5.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **Jetson Thor** edge device, through JupyterLab running on the Thor itself. **Every command in this notebook runs on the Thor**, not on your own computer.

Because the Thor is shared:

- use your own home folder and keep your files under it
- prefix any shared resource names with your account name using `${USER}`
- benchmark results are only fair when the device is not overloaded by others — note when the Thor is busy, and re-run if needed

This lab uses the latency logs you produced in the Edge AI lab. If you still have your `~/edgeAILab/data/*.jsonl` files, you can analyze them directly. If not, re-run the `latencyLogger.py` from that lab to regenerate them.

> **Inference environment:** Part 3 analyzes existing logs and needs only plain Python. But Parts 4, 5, and 7 **run inference** (`latencyLogger.py` imports `ultralytics`), so they need the Edge AI environment active — use the same kernel/container as Lab 5, and in any terminal re-activate it first:
> ```bash
> source ~/edgeAILab/venv/bin/activate
> ```
> or re-enter the inference container your instructor provided. The activation does not survive a new session, just like re-exporting `PORT` or `INFLUX_*` in the earlier labs.

---
## Part 1 · Set Up Your Working Folder

**[Notebook cell]** Create the lab identity. This lab exposes no network service, so no unique port matters here — `setupLab` still derives a default `PORT`, which simply goes unused. It also creates `~/benchmarkLab/` and writes `labEnv.sh` for the terminal scripts:

In [ ]:
import os

labConfig = setupLab(
    "benchmarkLab",
    extraEnv={"DEVICE_ID": f"{os.environ.get('USER', 'student01')}-thor"},
)

### Preflight · check your environment

In [ ]:
import os

preflight(
    [
        check("ultralytics importable (Parts 4, 5, 7)", pythonImportable("ultralytics"),
              hint="Use the same kernel as Lab 5 (instructor container or edge-ai venv). Part 3 works "
                   "without it, but the inference parts will not.",
              link="https://docs.ultralytics.com/quickstart/"),
        check("Lab 5 logger available", fileExists("~/edgeAILab/latencyLogger.py"),
              hint="Re-run Lab 5 Part 4's %%writefile latencyLogger.py cell - Parts 4, 5, and 7 here re-use it.",
              link="https://docs.ultralytics.com/modes/predict/"),
        check("Lab 5 data folder present", dirExists("~/edgeAILab/data"),
              hint="Run (or re-run) the Edge AI lab first - this lab analyzes its latency logs.",
              link="https://jsonlines.org/"),
        check("GPU telemetry on PATH (Part 4)", commandOnPath("nvidia-smi"),
              hint="tegrastats ships with JetPack. Without it, use the generic `top` fallback shown in Part 4.",
              link="https://docs.nvidia.com/jetson/archives/r36.3/DeveloperGuide/SD/PlatformPowerAndPerformance/JetsonOrinNanoSeriesJetsonOrinNxSeriesAndJetsonAgxOrinSeries.html"),
        check("git on PATH (Part 8, optional)", commandOnPath("git"),
              hint="Only the optional version-the-report step uses git; skip it if unavailable.",
              link="https://git-scm.com/doc"),
    ],
    infoRows=[
        ("your USER", os.environ.get("USER", "?")),
        ("DEVICE_ID", os.environ.get("DEVICE_ID", "?")),
    ],
)

**[Notebook cell]** Create the reports folder and copy in your latency logs from the Edge AI lab:

In [ ]:
!mkdir -p ~/benchmarkLab/reports
%cd ~/benchmarkLab
!cp ~/edgeAILab/data/latency*.jsonl ~/benchmarkLab/ 2>/dev/null || true
!ls -la *.jsonl 2>/dev/null || echo "no logs copied - re-run the Lab 5 logger first, then this cell"

If that copied nothing, re-run the logger from the Edge AI lab first (Lab 5, Part 4), then come back.

### Checkpoint · Part 1

In [ ]:
checkpoint(
    "Part 1 - benchmark inputs in place",
    [
        check("reports folder exists", dirExists("~/benchmarkLab/reports"),
              hint="Re-run the mkdir cell above.",
              link="https://docs.python.org/3/library/pathlib.html"),
        check("GPU latency log copied", fileNonEmpty("~/benchmarkLab/latencyGPU.jsonl", minLines=50),
              hint="Regenerate it in Lab 5 Part 4 (DEVICE=0 ... latencyGPU.jsonl), then re-run the copy cell.",
              link="https://jsonlines.org/"),
        check("CPU latency log copied", fileNonEmpty("~/benchmarkLab/latencyCPU.jsonl", minLines=50),
              hint="Regenerate it in Lab 5 Part 4 (DEVICE=cpu ... latencyCPU.jsonl), then re-run the copy cell.",
              link="https://jsonlines.org/"),
    ],
    successNote="Inputs staged - time to turn raw measurements into understanding.",
    docLink="https://jsonlines.org/",
)

---
## Part 2 · What a Benchmark Measures

A good benchmark reports more than an average. Key concepts:

```text
Latency percentiles -> p50 (median), p95, p99: most users feel the tail, not the mean
Throughput          -> results per second (FPS) under sustained load
Resource cost       -> CPU %, GPU %, memory, power used to hit that performance
Accuracy / quality  -> is the model still correct? speed is worthless if results are wrong
Repeatability       -> same conditions -> similar numbers, or the benchmark is meaningless
```

Why percentiles matter at the edge: a detector that averages 20 ms but spikes to 200 ms once a second can still miss a frame at the worst moment. The **p95** and **p99** (the slow tail) often decide whether a system is usable in real time.

---
## Part 3 · Compute Latency Statistics

Build an analyzer that reads a latency log and reports the full distribution.

**[Notebook cell]** Create `analyzeLatency.py` (the original lab uses `nano`; `%%writefile` does the same job):

In [ ]:
%%writefile analyzeLatency.py
import sys
import json


def percentile(sortedValues, pct):
    if not sortedValues:
        return 0.0
    k = (len(sortedValues) - 1) * (pct / 100.0)
    lowIndex = int(k)
    highIndex = min(lowIndex + 1, len(sortedValues) - 1)
    frac = k - lowIndex
    return sortedValues[lowIndex] + (sortedValues[highIndex] - sortedValues[lowIndex]) * frac


def analyze(path):
    endToEnd, inferenceTimes = [], []
    modelName = computeKind = None
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        endToEnd.append(record["latency_ms"]["end_to_end"])
        inferenceTimes.append(record["latency_ms"]["inference"])
        modelName = record.get("model", modelName)
        computeKind = record.get("compute", computeKind)

    if not endToEnd:
        print(f"{path}: no records")
        return

    endToEnd.sort()
    meanValue = sum(endToEnd) / len(endToEnd)
    print(f"\n=== {path} ===")
    print(f"model: {modelName}   compute: {computeKind}   runs: {len(endToEnd)}")
    print("end-to-end latency (ms):")
    print(f"  min  {endToEnd[0]:.2f}")
    print(f"  p50  {percentile(endToEnd, 50):.2f}")
    print(f"  mean {meanValue:.2f}")
    print(f"  p95  {percentile(endToEnd, 95):.2f}")
    print(f"  p99  {percentile(endToEnd, 99):.2f}")
    print(f"  max  {endToEnd[-1]:.2f}")
    print(f"mean model-inference (ms): {sum(inferenceTimes) / len(inferenceTimes):.2f}")
    print(f"throughput (FPS, sequential): {1000 / meanValue:.1f}")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("usage: python3 analyzeLatency.py <log.jsonl> [more.jsonl ...]")
        sys.exit(1)
    for path in sys.argv[1:]:
        analyze(path)

In [ ]:
# Preview the analyzer we just wrote.
showFile("~/benchmarkLab/analyzeLatency.py", maxLines=25)

**[Notebook cell]** Run it on your CPU and GPU logs together:

In [ ]:
!python3 analyzeLatency.py latencyCPU.jsonl latencyGPU.jsonl

**You should see:** one block per file showing min / p50 / mean / p95 / p99 / max latency, mean inference time, and an approximate FPS. The GPU block should show clearly lower latency than the CPU block.

> **New term · percentile (p50/p95/p99):** the value below which that share of runs fall. p50 is the median (typical run); p95/p99 are the slow "tail." At the edge the tail often decides whether a system is usable, because a single slow frame can miss a real-time deadline.

Compare the two. You can now talk about the **tail** (p95/p99), not just the average. The gap between p50 and p99 tells you how **consistent** each configuration is.

> **If it doesn't work:** `FileNotFoundError` means the log isn't in this folder — `ls *.jsonl`, and if empty, copy them from `~/edgeAILab/data/` (Part 1 cell) or re-run **Lab 5**'s logger (which needs the inference environment active).

### Checkpoint · Part 3

In [ ]:
import os

homeDir = os.path.expanduser("~")
analyzeCommand = (f"python3 {homeDir}/benchmarkLab/analyzeLatency.py "
                  f"{homeDir}/benchmarkLab/latencyGPU.jsonl")

checkpoint(
    "Part 3 - latency statistics",
    [
        check("analyzeLatency.py exists", fileExists("~/benchmarkLab/analyzeLatency.py"),
              hint="Re-run the %%writefile analyzeLatency.py cell above.",
              link="https://docs.python.org/3/library/json.html"),
        check("analyzer reports percentiles", outputContains(analyzeCommand, "p95"),
              hint="Run the analyzer cell above and read its error - usually a missing log (re-run Part 1's "
                   "copy cell) or an edited script (re-run the %%writefile cell).",
              link="https://docs.python.org/3/library/statistics.html"),
    ],
    successNote="You can now describe latency as a distribution - median, tail, and throughput.",
    docLink="https://docs.python.org/3/library/statistics.html",
)

---
## Part 4 · Measure the Resource Cost

Speed without resource context is half a benchmark. Capture device load **during** an inference run and pair it with the latency. This genuinely needs **two terminals** running at the same time — exactly what Jupyter terminals are for.

**[Notebook cell]** Write the two scripts:

In [ ]:
%%writefile runUnderLoad.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/benchmarkLab/labEnv.sh
cd ~/edgeAILab

echo "Sustained inference: 200 GPU runs -> ~/benchmarkLab/runUnderLoad.jsonl"
echo "(needs the Lab 5 inference environment - activate the venv/container first if this fails)"
DEVICE=0 RUNS=200 LOG_FILE=~/benchmarkLab/runUnderLoad.jsonl python3 latencyLogger.py

> **📟 On a Jetson:** same benchmark, platform-specific sampler. On a Jetson you capture the run with **`tegrastats --interval 1000`**; on this **DGX Spark (GB10)** with **`nvidia-smi dmon -d 1`**. Both emit one line per second so the downstream analysis is identical — only the collector changes. Knowing which tool a given edge box speaks is part of the job.

In [ ]:
%%writefile sampleTegrastats.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/benchmarkLab/labEnv.sh
cd ~/benchmarkLab

echo "Sampling GPU stats once a second -> reports/tegrastatsDuringRun.txt"
echo "Press Ctrl-C when the inference run in the other terminal finishes."
if command -v nvidia-smi >/dev/null; then
  nvidia-smi dmon -d 1 | tee reports/tegrastatsDuringRun.txt   # GB10: one line/sec
else
  tegrastats --interval 1000 | tee reports/tegrastatsDuringRun.txt   # Jetson
fi

In [ ]:
!chmod +x runUnderLoad.sh sampleTegrastats.sh

**[Terminal 1]** Start the sustained inference run:

```bash
~/benchmarkLab/runUnderLoad.sh
```

**[Terminal 2]** While it runs, sample device load into a file:

```bash
~/benchmarkLab/sampleTegrastats.sh
```

When the inference run finishes, stop the sampler with **Ctrl-C** in Terminal 2.

If `tegrastats` is not available, sample generic load instead in **[Terminal 2]**:

```bash
for i in $(seq 1 10); do top -b -n 1 | head -5; sleep 1; done > ~/benchmarkLab/reports/loadDuringRun.txt
```

**[Notebook cell]** Peek at the samples:

In [ ]:
!head -n 5 ~/benchmarkLab/reports/tegrastatsDuringRun.txt 2>/dev/null || \
 head -n 5 ~/benchmarkLab/reports/loadDuringRun.txt 2>/dev/null || \
 echo "no samples yet - run the two terminal scripts above first"

> The Thor is shared, so these numbers include all activity on the device, not only your job. Record whether the device was otherwise idle when you benchmarked — it is part of the result.

Now you can state performance **and** cost together: "this configuration delivered X FPS at Y% GPU and Z MB memory." That pairing is what makes a benchmark actionable.

### Checkpoint · Part 4

In [ ]:
import pathlib

def resourceSamplesProbe():
    reportsDir = pathlib.Path.home() / "benchmarkLab" / "reports"
    for sampleName in ("tegrastatsDuringRun.txt", "loadDuringRun.txt"):
        samplePath = reportsDir / sampleName
        if samplePath.is_file() and samplePath.stat().st_size > 0:
            lineCount = sum(1 for _ in samplePath.open(errors="replace"))
            return True, f"{sampleName}: {lineCount} sample line(s)"
    return False, "no resource samples found (tegrastatsDuringRun.txt or loadDuringRun.txt)"

checkpoint(
    "Part 4 - resource cost captured",
    [
        check("load-run latency log valid",
              jsonLinesValid("~/benchmarkLab/runUnderLoad.jsonl",
                             requiredKeys=["timestamp", "device_id", "model", "compute", "latency_ms"],
                             minRecords=100),
              hint="Run runUnderLoad.sh in Terminal 1 and let it finish (200 runs). If ultralytics failed to "
                   "import there, activate the Lab 5 venv/container in that terminal first.",
              link="https://jsonlines.org/"),
        check("resource samples recorded", resourceSamplesProbe,
              hint="Run sampleTegrastats.sh in Terminal 2 while the inference runs (or the `top` fallback), "
                   "then stop it with Ctrl-C.",
              link="https://docs.nvidia.com/jetson/archives/r36.3/DeveloperGuide/SD/PlatformPowerAndPerformance/JetsonOrinNanoSeriesJetsonOrinNxSeriesAndJetsonAgxOrinSeries.html"),
    ],
    successNote="Latency and resource cost captured together - performance with its price tag.",
    docLink="https://developer.nvidia.com/embedded/jetpack",
)

---
## Part 5 · Sweep Configurations

The real value of benchmarking is **comparing** options to choose the best one for the device. Sweep a few configurations and collect their logs under predictable names.

Run YOLO at two model sizes on the GPU (the nano and small models), plus CPU for contrast. **Each cell takes a minute or more — the CPU one the longest.** The noisy per-run lines are trimmed to the last two.

**[Notebook cell]**

In [ ]:
!cd ~/edgeAILab && DEVICE=0 MODEL=yolov8n.pt RUNS=100 \
  LOG_FILE=~/benchmarkLab/sweepNGPU.jsonl python3 latencyLogger.py | tail -n 2

In [ ]:
!cd ~/edgeAILab && DEVICE=0 MODEL=yolov8s.pt RUNS=100 \
  LOG_FILE=~/benchmarkLab/sweepSGPU.jsonl python3 latencyLogger.py | tail -n 2

In [ ]:
!cd ~/edgeAILab && DEVICE=cpu MODEL=yolov8n.pt RUNS=100 \
  LOG_FILE=~/benchmarkLab/sweepNCPU.jsonl python3 latencyLogger.py | tail -n 2

> `yolov8s.pt` is larger than `yolov8n.pt`: more accurate, but slower. That trade-off is exactly what the sweep should reveal. Your instructor may give you other models or input sizes to add.

**[Notebook cell]** Analyze the whole sweep at once:

In [ ]:
%cd ~/benchmarkLab
!python3 analyzeLatency.py sweepNGPU.jsonl sweepSGPU.jsonl sweepNCPU.jsonl

You changed **one variable at a time** (model size, then compute device), which is what makes a comparison fair. Note how the bigger model raises latency and how the CPU run collapses throughput.

### Checkpoint · Part 5

In [ ]:
sweepKeys = ["timestamp", "device_id", "model", "task", "compute", "run",
             "latency_ms", "num_detections"]

checkpoint(
    "Part 5 - configuration sweep",
    [
        check("sweepNGPU.jsonl valid (100+ runs)",
              jsonLinesValid("~/benchmarkLab/sweepNGPU.jsonl", requiredKeys=sweepKeys, minRecords=100),
              hint="Re-run the first sweep cell (yolov8n on GPU).",
              link="https://jsonlines.org/"),
        check("sweepSGPU.jsonl valid (100+ runs)",
              jsonLinesValid("~/benchmarkLab/sweepSGPU.jsonl", requiredKeys=sweepKeys, minRecords=100),
              hint="Re-run the second sweep cell (yolov8s on GPU) - its first run also downloads the "
                   "yolov8s.pt weights.",
              link="https://jsonlines.org/"),
        check("sweepNCPU.jsonl valid (100+ runs)",
              jsonLinesValid("~/benchmarkLab/sweepNCPU.jsonl", requiredKeys=sweepKeys, minRecords=100),
              hint="Re-run the third sweep cell (yolov8n on CPU) - it is slow by design, let it finish.",
              link="https://jsonlines.org/"),
    ],
    successNote="Three fair, one-variable-at-a-time configurations measured.",
    docLink="https://docs.ultralytics.com/models/yolov8/",
)

---
## Part 6 · Build a Results Table as CSV

A sweep is only useful if it is summarized so others can read it. Turn the logs into one CSV row per configuration.

**[Notebook cell]** Create `summarize.py` (the CSV column names like `p50_ms` are the report's data schema — keep them as written):

In [ ]:
%%writefile summarize.py
import sys
import json
import csv


def percentile(values, pct):
    orderedValues = sorted(values)
    if not orderedValues:
        return 0.0
    k = (len(orderedValues) - 1) * (pct / 100.0)
    lowIndex = int(k)
    highIndex = min(lowIndex + 1, len(orderedValues) - 1)
    return orderedValues[lowIndex] + (orderedValues[highIndex] - orderedValues[lowIndex]) * (k - lowIndex)


def summarize(path):
    endToEnd, modelName, computeKind = [], None, None
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        endToEnd.append(record["latency_ms"]["end_to_end"])
        modelName = record.get("model", modelName)
        computeKind = record.get("compute", computeKind)
    if not endToEnd:
        return None
    meanValue = sum(endToEnd) / len(endToEnd)
    return {
        "log": path,
        "model": modelName,
        "compute": computeKind,
        "runs": len(endToEnd),
        "p50_ms": round(percentile(endToEnd, 50), 2),
        "p95_ms": round(percentile(endToEnd, 95), 2),
        "p99_ms": round(percentile(endToEnd, 99), 2),
        "mean_ms": round(meanValue, 2),
        "fps": round(1000 / meanValue, 1),
    }


if __name__ == "__main__":
    rows = [row for row in (summarize(path) for path in sys.argv[1:]) if row]
    if not rows:
        print("no data")
        sys.exit(1)
    fields = ["log", "model", "compute", "runs", "p50_ms", "p95_ms", "p99_ms", "mean_ms", "fps"]
    with open("reports/results.csv", "w", newline="") as csvFile:
        writer = csv.DictWriter(csvFile, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    # also print a readable table
    print(" | ".join(fields))
    for row in rows:
        print(" | ".join(str(row[key]) for key in fields))

**[Notebook cell]** Run it across your sweep:

In [ ]:
!python3 summarize.py sweepNGPU.jsonl sweepSGPU.jsonl sweepNCPU.jsonl > reports/results.txt
!cat reports/results.txt

In [ ]:
# Preview the CSV we just produced.
showFile("~/benchmarkLab/reports/results.csv")

You now have a single comparison table: the heart of a benchmark report.

### Checkpoint · Part 6

In [ ]:
checkpoint(
    "Part 6 - results table",
    [
        check("summarize.py exists", fileExists("~/benchmarkLab/summarize.py"),
              hint="Re-run the %%writefile summarize.py cell above.",
              link="https://docs.python.org/3/library/csv.html"),
        check("results.csv has header + rows", fileNonEmpty("~/benchmarkLab/reports/results.csv", minLines=3),
              hint="Re-run the summarize cell - it needs the three sweep logs from Part 5 in ~/benchmarkLab.",
              link="https://docs.python.org/3/library/csv.html"),
        check("CSV carries the percentile columns", fileContains("~/benchmarkLab/reports/results.csv", "p95_ms"),
              hint="The schema changed - re-run the %%writefile summarize.py cell unchanged, then summarize again.",
              link="https://docs.python.org/3/library/csv.html"),
        check("readable results.txt saved", fileNonEmpty("~/benchmarkLab/reports/results.txt", minLines=2),
              hint="Re-run the `summarize.py ... > reports/results.txt` cell.",
              link="https://docs.python.org/3/library/csv.html"),
    ],
    successNote="One row per configuration - anyone can now read your comparison at a glance.",
    docLink="https://docs.python.org/3/library/csv.html",
)

---
## Part 7 · Test Sustained Load and Thermal Behavior

A short benchmark can hide a real edge problem: under **sustained** load the device heats up and may **throttle**, lowering clock speeds and slowing inference. Benchmark the steady state, not just the first burst.

**[Notebook cell]** Write the long-run script (400 runs is enough to expose drift without a long wait on the shared GPU — several minutes, so it runs in a terminal where you can watch it):

In [ ]:
%%writefile runSustained.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/benchmarkLab/labEnv.sh
cd ~/edgeAILab

echo "Sustained test: 400 GPU runs -> ~/benchmarkLab/sustained.jsonl (several minutes)"
echo "(needs the Lab 5 inference environment - activate the venv/container first if this fails)"
DEVICE=0 RUNS=400 LOG_FILE=~/benchmarkLab/sustained.jsonl python3 latencyLogger.py

In [ ]:
!chmod +x runSustained.sh

**[Terminal]** Run it and let it finish:

```bash
~/benchmarkLab/runSustained.sh
```

**[Notebook cell]** Analyze how latency changed from the start to the end of the run:

In [ ]:
import json
import pathlib

sustainedPath = pathlib.Path.home() / "benchmarkLab" / "sustained.jsonl"
if not sustainedPath.exists():
    showNote("sustained.jsonl not found - run runSustained.sh in a terminal first.", kind="warn")
else:
    records = [json.loads(line) for line in sustainedPath.open() if line.strip()]
    firstWindow = [record["latency_ms"]["end_to_end"] for record in records[:50]]
    lastWindow = [record["latency_ms"]["end_to_end"] for record in records[-50:]]
    firstMean = sum(firstWindow) / len(firstWindow)
    lastMean = sum(lastWindow) / len(lastWindow)
    print(f"first 50 mean: {firstMean:.2f} ms")
    print(f"last 50 mean:  {lastMean:.2f} ms")
    print("latency drift:",
          "increased (possible throttling)" if lastMean > firstMean * 1.1 else "stable")

If the last-50 average is meaningfully higher than the first-50, the device may be thermally throttling, or the shared GPU got busier. Either way, the **sustained** number is the honest one for an always-on edge deployment.

### Checkpoint · Part 7

In [ ]:
checkpoint(
    "Part 7 - sustained load",
    [
        check("sustained.jsonl valid (300+ runs)",
              jsonLinesValid("~/benchmarkLab/sustained.jsonl",
                             requiredKeys=["timestamp", "device_id", "model", "compute", "latency_ms"],
                             minRecords=300),
              hint="Run runSustained.sh in a terminal and let it complete all 400 runs.",
              link="https://jsonlines.org/"),
    ],
    successNote="Steady-state behavior measured - burst numbers can no longer fool you.",
    docLink="https://docs.nvidia.com/jetson/archives/r36.3/DeveloperGuide/SD/PlatformPowerAndPerformance/JetsonOrinNanoSeriesJetsonOrinNxSeriesAndJetsonAgxOrinSeries.html",
)

---
## Part 8 · Write the Benchmark Report

A benchmark that is not written down cannot be acted on. Capture the result and the conditions.

**[Notebook cell]** Write the report template — the cell will **not** overwrite the file if it already exists, so your edits are safe:

In [ ]:
import pathlib

reportPath = pathlib.Path.home() / "benchmarkLab" / "reports" / "benchmarkReport.md"
reportTemplate = '''# Edge Inference Benchmark Report

## Conditions
- Device: DGX Spark / GB10 (from `nvidia-smi -L` + `hostnamectl`; account: [your account])
- Date: [date]
- Device otherwise idle? [yes / no - other students active?]
- Power / limits: [from `nvidia-smi -q -d POWER`; a Jetson uses `nvpmodel -q`]

## Configurations Tested
[paste reports/results.txt table here]

## Findings
- Fastest configuration: [model + compute], p95 = [..] ms, [..] FPS
- Accuracy/quality note: [bigger model detected more / fewer; any misses]
- Sustained load: [latency stable / drifted up - throttling?]
- Resource cost: [GPU %, memory from reports/tegrastatsDuringRun.txt]

## Recommendation
- For this edge node, use [configuration] because [latency / accuracy / resource trade-off].
- This fits the resource budget from the System Architecture lab: [yes / no].
'''

if reportPath.exists():
    showNote(f"{reportPath} already exists - not overwriting your edits.", kind="info")
else:
    reportPath.write_text(reportTemplate)
    showNote(f"Template written to {reportPath}.", kind="ok")
showFile(reportPath)

Open `benchmarkLab/reports/benchmarkReport.md` from the Jupyter file browser (double-click it) and **fill in every `[...]` placeholder** from your runs — including pasting the `reports/results.txt` table into *Configurations Tested*.

**[Notebook cell]** Optionally version your benchmark with Git (set up in the first lab; each line is idempotent):

In [ ]:
!cd ~/benchmarkLab && git init -q 2>/dev/null || true
!cd ~/benchmarkLab && git add analyzeLatency.py summarize.py reports/ && \
  git -c user.name="${USER}" -c user.email="${USER}@edge-lab" \
  commit -q -m "Add edge inference benchmark and report" || echo "nothing new to commit"

A benchmark report is an engineering artifact: it states what was measured, under what conditions, and what to do about it.

### Checkpoint · Part 8

In [ ]:
checkpoint(
    "Part 8 - benchmark report",
    [
        check("Conditions section present",
              fileContains("~/benchmarkLab/reports/benchmarkReport.md", "## Conditions"),
              hint="Re-run the template cell above, and keep the four section headings when editing.",
              link="https://www.markdownguide.org/basic-syntax/"),
        check("Configurations Tested section present",
              fileContains("~/benchmarkLab/reports/benchmarkReport.md", "## Configurations Tested"),
              hint="Re-run the template cell above, and keep the four section headings when editing.",
              link="https://www.markdownguide.org/basic-syntax/"),
        check("Findings section present",
              fileContains("~/benchmarkLab/reports/benchmarkReport.md", "## Findings"),
              hint="Re-run the template cell above, and keep the four section headings when editing.",
              link="https://www.markdownguide.org/basic-syntax/"),
        check("Recommendation section present",
              fileContains("~/benchmarkLab/reports/benchmarkReport.md", "## Recommendation"),
              hint="Re-run the template cell above, and keep the four section headings when editing.",
              link="https://www.markdownguide.org/basic-syntax/"),
        check("results table pasted into the report",
              fileContains("~/benchmarkLab/reports/benchmarkReport.md", "p95_ms"),
              hint="Open the report in the Jupyter editor and paste the contents of reports/results.txt "
                   "under 'Configurations Tested'.",
              link="https://docs.python.org/3/library/csv.html"),
    ],
    successNote="Your benchmark is written down: measured, conditioned, and actionable.",
    docLink="https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository",
)

---
## Cleanup

Your reports and scripts are small. The large items are the latency logs and any model weights in the Edge AI lab folder.

If you committed the report with Git and want to keep it, push it to GitHub instead of deleting it. Only remove your own files.

**[Notebook cell]** Optional · remove your benchmark folder (keep it if you want the report — uncomment to run):

In [ ]:
# !rm -rf ~/benchmarkLab

---
### Lab scorecard

In [ ]:
labSummary("Benchmarking & Evaluations Lab")

---
## Part 9 · Connect to the Rest of the Course

You turned raw measurements into an evidence-based evaluation of the edge node:

```text
Inference latency logs            (Edge AI lab)
    -> Statistics + percentiles   (this lab)
    -> Resource cost + sweep      (this lab)
    -> Benchmark report           (this lab)
    -> Optimization               (next lab: make the winning config faster / smaller)
```

A real project benchmarks before and after every change, so improvements are proven, not assumed. The main idea: edge systems are judged by repeatable measurement of latency, throughput, resource cost, and accuracy — reported with the full distribution and the conditions it was measured under. That evidence is what the Optimization lab will try to improve.

---
## Key Terms

- **Benchmark:** a repeatable measurement of performance under controlled, recorded conditions.
- **Percentile (p50/p95/p99):** the latency below which that fraction of runs fall; p95/p99 are the slow "tail" that often decides real-time usability.
- **Latency vs. throughput:** latency is time per single result; throughput (FPS) is results per second.
- **Configuration sweep:** running the same benchmark while changing **one** variable at a time (model size, image size, CPU vs GPU) so comparisons are fair.
- **Resource cost:** the CPU/GPU/memory/power a workload consumes to hit its performance — half of any honest benchmark.
- **Thermal throttling:** sustained load heats the device, which lowers clock speeds, so steady-state performance can be worse than the first burst.
- **Sustained vs. burst performance:** a short test can look fast; the long-run (sustained) number is the honest one for an always-on edge deployment.